In [9]:
import os
import warnings
import pickle
import logging
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

DATA_PATH       = "/kaggle/input/datasets/quanghuylt/stock-price/stock_price.csv" 

REPORTS_DIR          = "reports/metrics/lstm/"
FIGURES_DIR          = "reports/figures/lstm/"
MODEL_DIR            = "models/lstm/"
VAL_PREDICTIONS_DIR  = "data/val/"
TEST_PREDICTIONS_DIR = "data/test/"

VAL_SUMMARY_METRICS  = os.path.join(REPORTS_DIR, "LSTM_val_summary_metrics.csv")
TEST_SUMMARY_METRICS = os.path.join(REPORTS_DIR, "LSTM_test_summary_metrics.csv")

SYMBOLS      = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
FEATURE_COLS = ["Open", "High", "Low", "Volume", "EMA_20", "RSI", "MACD"]
TARGET_COL   = "Target_Return"  # predicting return instead of absolute close

LOOKBACK     = 7
HIDDEN_SIZE  = 64  # slightly increased due to more features
NUM_LAYERS   = 2
BATCH_SIZE   = 16
EPOCHS       = 50
LR           = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for d in [REPORTS_DIR, FIGURES_DIR, MODEL_DIR, VAL_PREDICTIONS_DIR, TEST_PREDICTIONS_DIR]:
    os.makedirs(d, exist_ok=True)

In [ ]:
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values("Date").reset_index(drop=True)
    
    # 1. target: calculate percentage return
    df["Target_Return"] = df["Close"].pct_change()
    
    # 2. ema 20
    df["EMA_20"] = df["Close"].ewm(span=20, adjust=False).mean()
    
    # 3. rsi (14 days)
    delta = df["Close"].diff()
    up = delta.clip(lower=0)
    down = -1 * delta.clip(upper=0)
    ema_up = up.ewm(com=13, adjust=False).mean()
    ema_down = down.ewm(com=13, adjust=False).mean()
    rs = ema_up / ema_down
    df["RSI"] = 100 - (100 / (1 + rs))
    
    # 4. macd
    exp1 = df["Close"].ewm(span=12, adjust=False).mean()
    exp2 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = exp1 - exp2
    
    # drop rows with nan values created by rolling calculations
    df = df.dropna().reset_index(drop=True)
    return df

In [ ]:
def split_data_80_10_10(df: pd.DataFrame):
    n = len(df)
    train_end = int(n * 0.8)
    val_end   = int(n * 0.9)
    
    train_df = df.iloc[:train_end].reset_index(drop=True)
    val_df   = df.iloc[train_end:val_end].reset_index(drop=True)
    test_df  = df.iloc[val_end:].reset_index(drop=True)
    
    return train_df, val_df, test_df

def create_sequences(features, target, lookback):
    X, Y = [], []
    for i in range(len(features) - lookback):
        X.append(features[i : i + lookback])
        Y.append(target[i + lookback])
    return np.array(X), np.array(Y)

def prepare_tensors(train_df, val_df, test_df):
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    
    # strictly fit on train set only
    train_feat_scaled = scaler_x.fit_transform(train_df[FEATURE_COLS])
    train_targ_scaled = scaler_y.fit_transform(train_df[[TARGET_COL]])
    
    val_feat_scaled   = scaler_x.transform(val_df[FEATURE_COLS])
    val_targ_scaled   = scaler_y.transform(val_df[[TARGET_COL]])
    
    test_feat_scaled  = scaler_x.transform(test_df[FEATURE_COLS])
    test_targ_scaled  = scaler_y.transform(test_df[[TARGET_COL]])
    
    X_train, Y_train = create_sequences(train_feat_scaled, train_targ_scaled, LOOKBACK)
    X_val, Y_val     = create_sequences(val_feat_scaled, val_targ_scaled, LOOKBACK)
    X_test, Y_test   = create_sequences(test_feat_scaled, test_targ_scaled, LOOKBACK)
    
    tensors = {
        "train": (torch.tensor(X_train, dtype=torch.float32), torch.tensor(Y_train, dtype=torch.float32)),
        "val":   (torch.tensor(X_val, dtype=torch.float32), torch.tensor(Y_val, dtype=torch.float32)),
        "test":  (torch.tensor(X_test, dtype=torch.float32), torch.tensor(Y_test, dtype=torch.float32))
    }
    
    return tensors, scaler_x, scaler_y

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(x.device)
        c0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

def train_model(X_train, Y_train):
    dataset = TensorDataset(X_train, Y_train)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = LSTMModel(input_size=len(FEATURE_COLS), hidden_size=HIDDEN_SIZE,
                      num_layers=NUM_LAYERS, output_size=1).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            log.info(f"      epoch {epoch+1}/{EPOCHS}  loss={loss.item():.6f}")

    return model

def predict_and_reconstruct(model, X_tensor, scaler_y, original_df, split_name):
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_tensor.to(DEVICE)).cpu().numpy()
        
    # 1. inverse to get predicted returns
    preds_return = scaler_y.inverse_transform(preds_scaled).flatten()
    
    # 2. reconstruct actual close prices using p_t = p_{t-1} * (1 + r_t)
    # we align indices based on lookback
    prev_close   = original_df["Close"].values[LOOKBACK - 1 : -1]
    actual_close = original_df["Close"].values[LOOKBACK:]
    
    preds_close = prev_close * (1 + preds_return)
    
    result_df = pd.DataFrame({
        "Ticker": original_df["Ticker"].values[LOOKBACK:],
        "Date": original_df["Date"].values[LOOKBACK:],
        "Actual_Close": actual_close,
        "Predicted_Close": preds_close,
        "Actual_Return": original_df["Target_Return"].values[LOOKBACK:],
        "Predicted_Return": preds_return,
        "Split": split_name
    })
    return result_df

In [ ]:
def calc_metrics(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    # we calculate metrics based on the reconstructed price
    df = df.dropna(subset=["Actual_Close", "Predicted_Close"])
    
    y_true = df["Actual_Close"].values
    y_pred = df["Predicted_Close"].values

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8)))
    r2   = r2_score(y_true, y_pred)

    # directional accuracy (da) based on returns
    actual_dir  = np.sign(df["Actual_Return"].values)
    pred_dir    = np.sign(df["Predicted_Return"].values)
    da = np.mean(actual_dir == pred_dir) if len(actual_dir) > 0 else np.nan
    
    # trend prediction accuracy (tpa)
    mask = actual_dir != 0
    tpa = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    # volatility rmse (v-rmse)
    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "Ticker": ticker,
        "MSE": mse, "MAE": mae, "MAPE": mape, 
        "RMSE": rmse, "R2": r2, "DA": da,
        "TPA": tpa, "V-RMSE": v_rmse
    }])

def plot_results(val_df, test_df, ticker: str, save_dir: str):
    sns.set(style="whitegrid")
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    plot_df = pd.concat([val_df, test_df])
    
    axes[0].plot(plot_df["Date"], plot_df["Actual_Close"], label="Actual Close", color="#1f77b4", linewidth=2)
    axes[0].plot(val_df["Date"], val_df["Predicted_Close"], label="Val Predicted", color="#ff7f0e", linestyle="--", linewidth=1.5)
    axes[0].plot(test_df["Date"], test_df["Predicted_Close"], label="Test Predicted", color="#d62728", linestyle="--", linewidth=1.5)
    
    axes[0].set_title(f"LSTM Predicted Close: {ticker}", fontsize=15)
    axes[0].set_xlabel("Date", fontsize=12)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="best")
    axes[0].tick_params(axis="x", rotation=30)

    error = plot_df["Actual_Close"] - plot_df["Predicted_Close"]
    colors = np.where(error >= 0, "#2ca02c", "#d62728")
    axes[1].bar(plot_df["Date"], error, color=colors, alpha=0.6, width=1.5)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Prediction Error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{ticker}_val_test.png"), bbox_inches="tight", dpi=150)
    plt.close()

In [ ]:
def run_pipeline():
    if not DATA_PATH or not os.path.exists(DATA_PATH):
        log.error("please fill in data_path to run the script.")
        return

    df_full = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    
    val_metrics_list = []
    test_metrics_list = []

    for ticker in SYMBOLS:
        log.info(f"========== processing {ticker} ==========")
        df_ticker = df_full[df_full["Ticker"] == ticker]
        
        train_df, val_df, test_df = split_data_80_10_10(df_ticker)
        train_df = add_technical_indicators(train_df)
        val_df   = add_technical_indicators(val_df)
        test_df  = add_technical_indicators(test_df)
        log.info(f"  split rows - train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

        # 3. prepare tensors and scale
        tensors, scaler_x, scaler_y = prepare_tensors(train_df, val_df, test_df)
        X_train, Y_train = tensors["train"]
        X_val, Y_val     = tensors["val"]
        X_test, Y_test   = tensors["test"]

        # 4. train model
        log.info("  training model...")
        model = train_model(X_train, Y_train)

        # 5. predict and reconstruct price
        val_res_df  = predict_and_reconstruct(model, X_val, scaler_y, val_df, "Validation")
        test_res_df = predict_and_reconstruct(model, X_test, scaler_y, test_df, "Test")

        val_res_df.to_csv(os.path.join(VAL_PREDICTIONS_DIR, f"LSTM_{ticker}_val.csv"), index=False)
        test_res_df.to_csv(os.path.join(TEST_PREDICTIONS_DIR, f"LSTM_{ticker}_test.csv"), index=False)

        # 6. calculate metrics
        val_metrics = calc_metrics(val_res_df, ticker)
        test_metrics = calc_metrics(test_res_df, ticker)
        
        val_metrics_list.append(val_metrics)
        test_metrics_list.append(test_metrics)

        log.info(f"  VAL MAE: {val_metrics['MAE'].iloc[0]:.4f} | R2: {val_metrics['R2'].iloc[0]:.4f} | TPA: {val_metrics['TPA'].iloc[0]:.4f}")
        log.info(f"  TEST MAE: {test_metrics['MAE'].iloc[0]:.4f} | R2: {test_metrics['R2'].iloc[0]:.4f} | TPA: {test_metrics['TPA'].iloc[0]:.4f}")

        # 7. plot
        plot_results(val_res_df, test_res_df, ticker, FIGURES_DIR)

        # 8. save models
        with open(os.path.join(MODEL_DIR, f"lstm_{ticker}.pkl"), "wb") as f:
            pickle.dump({
                "model_state":  model.state_dict(),
                "scaler_x":     scaler_x,
                "scaler_y":     scaler_y,
                "ticker":       ticker
            }, f)
            
        torch.cuda.empty_cache()

    if val_metrics_list and test_metrics_list:
        summary_val_df = pd.concat(val_metrics_list, ignore_index=True)
        summary_test_df = pd.concat(test_metrics_list, ignore_index=True)
        
        summary_val_df.to_csv(VAL_SUMMARY_METRICS, index=False)
        summary_test_df.to_csv(TEST_SUMMARY_METRICS, index=False)
        
        log.info(f"saved val metrics: {VAL_SUMMARY_METRICS}")
        log.info(f"saved test metrics: {TEST_SUMMARY_METRICS}")

if __name__ == "__main__":
    run_pipeline()

12:50:48  INFO      ========== processing VNM ==========
12:50:48  INFO        split rows - train: 3258, val: 406, test: 407
12:50:48  INFO        training model...
12:50:53  INFO            epoch 10/50  loss=0.444310
12:50:58  INFO            epoch 20/50  loss=0.493758
12:51:03  INFO            epoch 30/50  loss=0.436450
12:51:08  INFO            epoch 40/50  loss=0.262149
12:51:13  INFO            epoch 50/50  loss=0.193667
12:51:13  INFO        VAL MAE: 0.7035 | R2: 0.9107 | TPA: 0.5176
12:51:13  INFO        TEST MAE: 0.9268 | R2: 0.9155 | TPA: 0.4638
12:51:15  INFO      ========== processing FPT ==========
12:51:15  INFO        split rows - train: 3257, val: 406, test: 407
12:51:15  INFO        training model...
12:51:20  INFO            epoch 10/50  loss=0.067204
12:51:25  INFO            epoch 20/50  loss=0.023802
12:51:30  INFO            epoch 30/50  loss=0.022808
12:51:35  INFO            epoch 40/50  loss=0.022657
12:51:40  INFO            epoch 50/50  loss=0.058915
12:51:40 